# Análisis de Incidentes de Ciberseguridad: Impacto Financiero y en el Mercado de Valores

Este proyecto analiza el impacto de incidentes de ciberseguridad desde tres perspectivas:
- Incidentes registrados
- Impacto financiero
- Impacto en el mercado

Stack utilizado:
- Python (Pandas) para ETL
- Power BI para visualización

### 1. Carga de librerias
#### Descripción:
#### En esta sección se cargan los tres datasets principales del proyecto:
#### incidents_master, financial_impact y market_impact.
#### Se utilizan las librerías pandas para la lectura de archivos CSV,
#### permitiendo trabajar posteriormente con los datos de forma estructurada.

In [1]:
import pandas as pd
import numpy as np

### 2. Carga de datos
En esta sección se cargan los datasets principales del proyecto.

In [2]:
incidents = pd.read_csv('../data/incidents_master.csv')
financial = pd.read_csv('../data/financial_impact (1).csv')
market = pd.read_csv('../data/market_impact.csv')

### 3. Exploración Inicial
### Descripción:
##### Se realiza una exploración preliminar de los datos para entender su estructura, tipos de variables, presencia de valores nulos y posibles inconsistencias.
##### Esto permite identificar problemas de calidad de datos antes de la transformación.

In [3]:
print(incidents.shape)
print(financial.shape)
print(market.shape)

incidents.head()

(850, 32)
(778, 19)
(358, 31)


,incident_id,company_name,company_revenue_usd,country_hq,industry_primary,industry_secondary,employee_count,is_public_company,stock_ticker,incident_date,...,data_source_primary,data_source_secondary,data_source_type,confidence_tier,quality_score,quality_grade,review_flag,notes,created_at,updated_at
0,2021-0508-001,Quantum Asset Assurance Group Inc.,1.343769e+09,US,52,54,3940,True,QAA,2021-05-08,...,https://www.sec.gov/cgi-bin/browse-edgar?actio...,https://www.quantum-asset-assurance-g.com/news...,sec_filing,1,97.89,Gold,NaN,Multiple subsidiaries affected across 6 jurisd...,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z
1,2025-1211-001,Quantum Apex Ventures Ltd.,6.367059e+07,GB,51,NaN,250,False,NaN,2025-12-11,...,https://www.theregister.com/2025/06/14/quantum...,https://blog.talosintelligence.com/2025/09/qua...,verified_media,3,86.74,Gold,NaN,NaN,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z
2,2023-0115-001,BitWire Innovations Corp.,2.480619e+10,US,51,NaN,71369,True,BITW,2023-01-15,...,https://therecord.media/2023/12/10/bitwire-inn...,https://blog.talosintelligence.com/2023/08/bit...,verified_media,4,83.74,Silver,NaN,NaN,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z
3,2021-0315-001,Sterling Forge Markets Holdings Inc.,1.398259e+08,US,44-45,NaN,912,True,SFM,2021-03-15,...,https://www.sterling-forge-markets-ho.com/news...,https://www.mandiant.com/resources/blog/sterli...,company_pr,2,94.51,Gold,NaN,NaN,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z
4,2021-1204-001,Sierra Quantum Innovations Group Inc.,6.916977e+08,US,51,NaN,1662,True,SQI,2021-12-04,...,https://www.sierra-quantum-innovation.com/news...,NaN,company_pr,2,79.82,Silver,NaN,NaN,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z


### 3.1. Mostramos información de los dataframes
#### Aqui determinamos el número de filas, las columnas, valores no nulos y tipo de dato (dtype)

In [4]:
incidents.info()
financial.info()
market.info()

<class 'pandas.DataFrame'>
RangeIndex: 850 entries, 0 to 849
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   incident_id               850 non-null    str    
 1   company_name              850 non-null    str    
 2   company_revenue_usd       850 non-null    float64
 3   country_hq                850 non-null    str    
 4   industry_primary          850 non-null    str    
 5   industry_secondary        153 non-null    str    
 6   employee_count            850 non-null    int64  
 7   is_public_company         850 non-null    bool   
 8   stock_ticker              412 non-null    str    
 9   incident_date             850 non-null    str    
 10  incident_date_estimated   850 non-null    bool   
 11  discovery_date            850 non-null    str    
 12  disclosure_date           850 non-null    str    
 13  attack_vector_primary     850 non-null    str    
 14  attack_vector_seconda

### 3.2. Analizamos la cantidad de valores nulos en cada columna del dataset de incidentes
#### - Identificamos campos incompletos (ej: days_to_price_recovery)
#### - Detectamos qué variables requieren tratamiento (relleno, categorización o eliminación)

In [5]:
incidents.isnull().sum()
financial.isnull().sum()
market.isnull().sum()

incident_id                          0
stock_ticker                         0
price_7d_before                      0
price_disclosure_day                 0
price_1d_after                       0
price_7d_after                       0
price_30d_after                      0
volume_avg_30d_baseline              0
volume_disclosure_day                0
sector_index                         0
sector_return_same_period            0
abnormal_return_1d                   0
abnormal_return_7d                   0
abnormal_return_30d                  0
car_neg1_to_pos1                     0
car_0_to_7                           0
car_0_to_30                          0
car_0_to_90                          0
t_statistic_1d                       0
p_value_1d                           0
t_statistic_30d                      0
p_value_30d                          0
earnings_announcement_within_7d      0
market_cap_at_disclosure             0
volume_ratio_disclosure              0
pre_incident_volatility_3

### 4. Limpieza de datos
Se gestionan valores nulos en variables financieras.

In [6]:
financial['ransom_paid_usd'] = financial['ransom_paid_usd'].fillna(0)
financial['regulatory_fine_usd'] = financial['regulatory_fine_usd'].fillna(0)
financial['legal_fees_usd'] = financial['legal_fees_usd'].fillna(0)

### 5. Variables derivadas
Se crean nuevas variables para enriquecer el análisis:
- Tamaño de empresa
- Estado de recuperación

In [7]:
def categorizar_empresa(valor):
    if valor < 1e9:
        return 'Pequeña'
    elif valor < 1e12:
        return 'Mediana'
    else:
        return 'Grande'

incidents['company_size'] = incidents['company_revenue_usd'].apply(categorizar_empresa)

In [8]:
def estado_recuperacion(x):
    if pd.isna(x):
        return 'No recuperado'
    elif x == 0:
        return 'Recuperación inmediata'
    elif x <= 30:
        return 'Recuperación rápida'
    else:
        return 'Recuperación lenta'

market['estado_recuperacion'] = market['days_to_price_recovery'].apply(estado_recuperacion)

### 6. Cálculo de métricas
Se calculan indicadores clave como:
- Total de registros comprometidos
- Coste total de incidentes

In [9]:
total_records = incidents['data_compromised_records'].sum()
print('Total registros comprometidos:', total_records)

Total registros comprometidos: 1630493948.0


In [10]:
financial['total_cost'] = (
    financial['ransom_paid_usd'] +
    financial['regulatory_fine_usd'] +
    financial['legal_fees_usd']
)

total_cost = financial['total_cost'].sum()
print('Coste total:', total_cost)

Coste total: 6447676741.719999


### 7. Integración de datos
Se combinan los datasets en uno solo mediante incident_id.

In [11]:
df = incidents.merge(financial, on='incident_id', how='left')
df = df.merge(market, on='incident_id', how='left')

In [12]:
df.head()

,incident_id,company_name,company_revenue_usd,country_hq,industry_primary,industry_secondary,employee_count,is_public_company,stock_ticker_x,incident_date,...,earnings_announcement_within_7d,market_cap_at_disclosure,volume_ratio_disclosure,pre_incident_volatility_30d,post_incident_volatility_30d,days_to_price_recovery,notes,created_at,updated_at,estado_recuperacion
0,2021-0508-001,Quantum Asset Assurance Group Inc.,1.343769e+09,US,52,54,3940,True,QAA,2021-05-08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-1211-001,Quantum Apex Ventures Ltd.,6.367059e+07,GB,51,NaN,250,False,NaN,2025-12-11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-0115-001,BitWire Innovations Corp.,2.480619e+10,US,51,NaN,71369,True,BITW,2023-01-15,...,True,1.181988e+11,2.4652,0.027705,0.052161,255.0,NaN,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z,Recuperación lenta
3,2021-0315-001,Sterling Forge Markets Holdings Inc.,1.398259e+08,US,44-45,NaN,912,True,SFM,2021-03-15,...,False,6.489114e+08,3.0973,0.017116,0.027638,324.0,NaN,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z,Recuperación lenta
4,2021-1204-001,Sierra Quantum Innovations Group Inc.,6.916977e+08,US,51,NaN,1662,True,SQI,2021-12-04,...,False,4.735164e+09,1.5348,0.038209,0.045756,19.0,NaN,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z,Recuperación rápida


## Conclusiones

- Los incidentes de ciberseguridad generan impactos económicos significativos.
- Las empresas grandes presentan mayores pérdidas absolutas.
- El tiempo de recuperación en el mercado varía considerablemente.
- La integración de datos permite una visión completa del impacto.

### 8. Transformaciones equivalentes a Power BI
#### Descripción:
A continuación se replican en Python los pasos principales aplicados en Power Query con lenguaje M.
Esto permite dejar documentado el ETL también en el notebook y mantener coherencia entre Python y Power BI.


In [13]:
# Copias de trabajo para no modificar los dataframes originales
incidents_py = incidents.copy()
financial_py = financial.copy()
market_py = market.copy()

# Conversión de tipos similar a Power BI
for col in ['incident_date', 'discovery_date', 'disclosure_date']:
    if col in incidents_py.columns:
        incidents_py[col] = pd.to_datetime(incidents_py[col], errors='coerce')

for col in ['created_at', 'updated_at']:
    if col in incidents_py.columns:
        incidents_py[col] = pd.to_datetime(incidents_py[col], errors='coerce')

for col in ['company_revenue_usd', 'employee_count', 'data_compromised_records', 'downtime_hours', 'quality_score']:
    if col in incidents_py.columns:
        incidents_py[col] = pd.to_numeric(incidents_py[col], errors='coerce')

for col in ['direct_loss_usd', 'ransom_paid_usd', 'recovery_cost_usd', 'regulatory_fine_usd', 'insurance_payout_usd', 'total_loss_usd', 'legal_fees_usd']:
    if col in financial_py.columns:
        financial_py[col] = pd.to_numeric(financial_py[col], errors='coerce')

for col in ['price_disclosure_day', 'sector_return_same_period', 'abnormal_return_1d', 'abnormal_return_30d', 'days_to_price_recovery']:
    if col in market_py.columns:
        market_py[col] = pd.to_numeric(market_py[col], errors='coerce')


In [14]:
# Eliminación de columnas de bajo valor analítico, igual que en Power BI
columnas_quitar_incidents = [
    'created_at',
    'updated_at',
    'data_source_secondary',
    'data_source_type',
    'data_source_primary',
    'confidence_tier',
    'quality_grade',
    'review_flag',
    'notes'
]

incidents_py = incidents_py.drop(columns=[c for c in columnas_quitar_incidents if c in incidents_py.columns], errors='ignore')
incidents_py.head()


,incident_id,company_name,company_revenue_usd,country_hq,industry_primary,industry_secondary,employee_count,is_public_company,stock_ticker,incident_date,...,attack_vector_secondary,attack_chain,attributed_group,attribution_confidence,data_compromised_records,data_type,systems_affected,downtime_hours,quality_score,company_size
0,2021-0508-001,Quantum Asset Assurance Group Inc.,1.343769e+09,US,52,54,3940,True,QAA,2021-05-08,...,NaN,Phishing email with weaponized attachment → cr...,NoName057,suspected,26888.0,mixed,"payment_systems,HR_systems",16.52,97.89,Mediana
1,2025-1211-001,Quantum Apex Ventures Ltd.,6.367059e+07,GB,51,NaN,250,False,NaN,2025-12-11,...,NaN,NaN,FIN11,probable,41426.0,mixed,"file_servers,backup_systems,HR_systems,ERP",NaN,86.74,Pequeña
2,2023-0115-001,BitWire Innovations Corp.,2.480619e+10,US,51,NaN,71369,True,BITW,2023-01-15,...,ddos,Drive-by download from compromised ad network ...,NaN,NaN,NaN,NaN,HR_systems,NaN,83.74,Mediana
3,2021-0315-001,Sterling Forge Markets Holdings Inc.,1.398259e+08,US,44-45,NaN,912,True,SFM,2021-03-15,...,NaN,Spear-phishing campaign targeting finance team...,NaN,NaN,1107796.0,mixed,"HR_systems,CRM,VPN",NaN,94.51,Pequeña
4,2021-1204-001,Sierra Quantum Innovations Group Inc.,6.916977e+08,US,51,NaN,1662,True,SQI,2021-12-04,...,NaN,NaN,NaN,NaN,621233.0,mixed,"SCADA,VPN",NaN,79.82,Pequeña


In [15]:
# Variables temporales derivadas
incidents_py['incident_year'] = incidents_py['incident_date'].dt.year
incidents_py['incident_month'] = incidents_py['incident_date'].dt.month
incidents_py['incident_quarter'] = incidents_py['incident_date'].dt.quarter
incidents_py['incident_quarter_label'] = incidents_py['incident_quarter'].apply(
    lambda x: f'Q{int(x)}' if pd.notna(x) else np.nan
)
incidents_py['incident_year_quarter'] = incidents_py.apply(
    lambda row: f"{int(row['incident_year'])} Q{int(row['incident_quarter'])}"
    if pd.notna(row['incident_year']) and pd.notna(row['incident_quarter'])
    else np.nan,
    axis=1
)


In [16]:
# Categorizaciones equivalentes a Power BI

def categorizar_tamano_ingresos(valor):
    if pd.isna(valor):
        return 'Sin dato'
    elif valor < 1_000_000_000:
        return '< 1B'
    elif valor < 10_000_000_000:
        return '1B–10B'
    elif valor < 100_000_000_000:
        return '10B–100B'
    elif valor < 1_000_000_000_000:
        return '100B–1T'
    return '> 1T'


def categorizar_impacto_datos(valor):
    if pd.isna(valor):
        return 'Sin dato'
    elif valor < 1_000:
        return 'Bajo'
    elif valor < 100_000:
        return 'Medio'
    return 'Alto'


def categorizar_impacto_downtime(valor):
    if pd.isna(valor):
        return 'Sin dato'
    elif valor < 24:
        return 'Bajo'
    elif valor < 72:
        return 'Medio'
    return 'Alto'


def categorizar_tamano_empleados(valor):
    if pd.isna(valor):
        return 'Sin dato'
    elif valor < 50:
        return 'Pequeña'
    elif valor < 250:
        return 'Mediana'
    elif valor < 1000:
        return 'Grande'
    return 'Muy grande'

incidents_py['company_size_revenue'] = incidents_py['company_revenue_usd'].apply(categorizar_tamano_ingresos)
incidents_py['data_impact_category'] = incidents_py['data_compromised_records'].apply(categorizar_impacto_datos)
incidents_py['downtime_impact_category'] = incidents_py['downtime_hours'].apply(categorizar_impacto_downtime)
incidents_py['company_size_employee'] = incidents_py['employee_count'].apply(categorizar_tamano_empleados)


In [17]:
# Limpieza de texto y variables auxiliares

def texto_valido(valor):
    if pd.isna(valor):
        return False
    valor = str(valor).strip()
    return valor != '' and valor.upper() != 'NULL'


def limpiar_texto(valor):
    if pd.isna(valor):
        return valor
    return str(valor).replace('', '').strip()

if 'industry_primary' in incidents_py.columns:
    incidents_py['industry_primary'] = incidents_py['industry_primary'].apply(limpiar_texto)

incidents_py['tipo_industria'] = incidents_py.apply(
    lambda row: 'Primaria y Secundaria'
    if texto_valido(row.get('industry_primary')) and texto_valido(row.get('industry_secondary'))
    else 'Solo Primaria'
    if texto_valido(row.get('industry_primary'))
    else 'Solo Secundaria'
    if texto_valido(row.get('industry_secondary'))
    else 'Sin industria',
    axis=1
)

incidents_py['attack_vector_secondary_label'] = incidents_py['attack_vector_secondary'].apply(
    lambda x: x if texto_valido(x) else 'Sin vector secundario'
)


In [18]:
# Preparación del dataset financiero
columnas_financieras = [
    'incident_id',
    'direct_loss_usd',
    'ransom_paid_usd',
    'ransom_paid_flag',
    'recovery_cost_usd',
    'regulatory_fine_usd',
    'insurance_payout_usd',
    'total_loss_usd'
]

financial_sel = financial_py[[c for c in columnas_financieras if c in financial_py.columns]].copy()

for col in ['ransom_paid_usd', 'regulatory_fine_usd']:
    if col in financial_sel.columns:
        financial_sel[col] = financial_sel[col].fillna(0)

financial_sel.head()


,incident_id,direct_loss_usd,ransom_paid_usd,recovery_cost_usd,regulatory_fine_usd,insurance_payout_usd,total_loss_usd
0,2021-0508-001,12600000.00,0.0,9455354.49,90695.25,6756288.97,24642595.67
1,2025-1211-001,7640471.18,0.0,5857150.47,0.00,2691027.33,15306810.06
2,2023-0115-001,34881599.59,0.0,26404111.95,0.00,31759649.99,71616414.97
3,2021-0315-001,4682151.47,0.0,3642946.48,0.00,1772460.33,9354133.80
4,2021-1204-001,2684607.92,0.0,2574871.33,0.00,NaN,5466301.48


In [19]:
# Preparación del dataset de mercado

def clasificar_estado_recuperacion(x):
    if pd.isna(x):
        return 'No recuperado / No aplica'
    elif x == 0:
        return 'Recuperación inmediata'
    elif x <= 30:
        return 'Recuperación rápida'
    return 'Recuperación lenta'


def clasificar_rango_recuperacion(x):
    if pd.isna(x):
        return 'Sin dato'
    elif x == 0:
        return '0 días'
    elif x <= 7:
        return '1-7 días'
    elif x <= 30:
        return '8-30 días'
    return '>30 días'

if 'days_to_price_recovery' in market_py.columns:
    if 'estado_recuperacion' not in market_py.columns:
        market_py['estado_recuperacion'] = market_py['days_to_price_recovery'].apply(clasificar_estado_recuperacion)
    if 'rango_recuperacion' not in market_py.columns:
        market_py['rango_recuperacion'] = market_py['days_to_price_recovery'].apply(clasificar_rango_recuperacion)

columnas_market = [
    'incident_id',
    'price_disclosure_day',
    'sector_return_same_period',
    'abnormal_return_1d',
    'abnormal_return_30d',
    'days_to_price_recovery',
    'rango_recuperacion',
    'estado_recuperacion'
]

market_sel = market_py[[c for c in columnas_market if c in market_py.columns]].copy()
market_sel.head()


,incident_id,price_disclosure_day,sector_return_same_period,abnormal_return_1d,abnormal_return_30d,days_to_price_recovery,rango_recuperacion,estado_recuperacion
0,2023-0115-001,251.95,0.018777,-0.027129,-0.020181,255.0,>30 días,Recuperación lenta
1,2021-0315-001,9.09,0.002086,-0.039530,-0.045737,324.0,>30 días,Recuperación lenta
2,2021-1204-001,13.36,-0.014242,-0.020519,-0.014270,19.0,8-30 días,Recuperación rápida
3,2021-0213-001,5.43,-0.006342,-0.029562,-0.010566,24.0,8-30 días,Recuperación rápida
4,2025-0529-001,441.08,-0.017191,-0.069433,-0.026270,313.0,>30 días,Recuperación lenta


### 9. Integración final de datos
Se integran los tres datasets con `incident_id` para construir una tabla maestra lista para análisis en Python o exportación a Power BI.


In [20]:
# Merge final equivalente al modelo consolidado
# Nota: en Power BI aparece una combinación duplicada con financial_impact.
# Aquí se deja una sola integración para evitar columnas repetidas innecesarias.

df_final = incidents_py.merge(financial_sel, on='incident_id', how='left')
df_final = df_final.merge(market_sel, on='incident_id', how='left')

orden_columnas = [
    'incident_id', 'company_name', 'company_revenue_usd', 'company_size_revenue',
    'country_hq', 'industry_primary', 'industry_secondary', 'tipo_industria',
    'employee_count', 'company_size_employee', 'is_public_company', 'stock_ticker',
    'incident_date', 'incident_year', 'incident_month', 'incident_quarter',
    'incident_quarter_label', 'incident_year_quarter', 'incident_date_estimated',
    'discovery_date', 'disclosure_date', 'attack_vector_primary',
    'attack_vector_secondary', 'attack_vector_secondary_label', 'attack_chain',
    'attributed_group', 'attribution_confidence', 'data_compromised_records',
    'data_impact_category', 'data_type', 'systems_affected', 'downtime_hours',
    'downtime_impact_category', 'quality_score', 'direct_loss_usd',
    'ransom_paid_usd', 'ransom_paid_flag', 'recovery_cost_usd',
    'regulatory_fine_usd', 'insurance_payout_usd', 'total_loss_usd',
    'price_disclosure_day', 'sector_return_same_period', 'abnormal_return_1d',
    'abnormal_return_30d', 'days_to_price_recovery', 'rango_recuperacion',
    'estado_recuperacion'
]

columnas_presentes = [c for c in orden_columnas if c in df_final.columns]
resto_columnas = [c for c in df_final.columns if c not in columnas_presentes]
df_final = df_final[columnas_presentes + resto_columnas]

df_final.head()


,incident_id,company_name,company_revenue_usd,company_size_revenue,country_hq,industry_primary,industry_secondary,tipo_industria,employee_count,company_size_employee,...,insurance_payout_usd,total_loss_usd,price_disclosure_day,sector_return_same_period,abnormal_return_1d,abnormal_return_30d,days_to_price_recovery,rango_recuperacion,estado_recuperacion,company_size
0,2021-0508-001,Quantum Asset Assurance Group Inc.,1.343769e+09,1B–10B,US,52,54,Primaria y Secundaria,3940,Muy grande,...,6756288.97,24642595.67,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Mediana
1,2025-1211-001,Quantum Apex Ventures Ltd.,6.367059e+07,< 1B,GB,51,NaN,Solo Primaria,250,Grande,...,2691027.33,15306810.06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pequeña
2,2023-0115-001,BitWire Innovations Corp.,2.480619e+10,10B–100B,US,51,NaN,Solo Primaria,71369,Muy grande,...,31759649.99,71616414.97,251.95,0.018777,-0.027129,-0.020181,255.0,>30 días,Recuperación lenta,Mediana
3,2021-0315-001,Sterling Forge Markets Holdings Inc.,1.398259e+08,< 1B,US,44-45,NaN,Solo Primaria,912,Grande,...,1772460.33,9354133.80,9.09,0.002086,-0.039530,-0.045737,324.0,>30 días,Recuperación lenta,Pequeña
4,2021-1204-001,Sierra Quantum Innovations Group Inc.,6.916977e+08,< 1B,US,51,NaN,Solo Primaria,1662,Muy grande,...,NaN,5466301.48,13.36,-0.014242,-0.020519,-0.014270,19.0,8-30 días,Recuperación rápida,Pequeña


In [21]:
# Validación rápida del resultado final
print('Dimensión final:', df_final.shape)
print(df_final.columns.tolist())


Dimensión final: (850, 48)
['incident_id', 'company_name', 'company_revenue_usd', 'company_size_revenue', 'country_hq', 'industry_primary', 'industry_secondary', 'tipo_industria', 'employee_count', 'company_size_employee', 'is_public_company', 'stock_ticker', 'incident_date', 'incident_year', 'incident_month', 'incident_quarter', 'incident_quarter_label', 'incident_year_quarter', 'incident_date_estimated', 'discovery_date', 'disclosure_date', 'attack_vector_primary', 'attack_vector_secondary', 'attack_vector_secondary_label', 'attack_chain', 'attributed_group', 'attribution_confidence', 'data_compromised_records', 'data_impact_category', 'data_type', 'systems_affected', 'downtime_hours', 'downtime_impact_category', 'quality_score', 'direct_loss_usd', 'ransom_paid_usd', 'recovery_cost_usd', 'regulatory_fine_usd', 'insurance_payout_usd', 'total_loss_usd', 'price_disclosure_day', 'sector_return_same_period', 'abnormal_return_1d', 'abnormal_return_30d', 'days_to_price_recovery', 'rango_rec

## Conclusión técnica del ETL
- Se documentó en Python el mismo flujo principal aplicado en Power BI.
- Se derivaron variables temporales, categóricas y de impacto para enriquecer el análisis.
- Se consolidaron los tres CSV en una tabla maestra mediante `incident_id`.
- El dataset final queda preparado para análisis exploratorio, exportación o conexión con Power BI.


In [22]:
# Exportar dataset final para Power BI
output_path = '../data/df_final_powerbi.csv'
df_final.to_csv(output_path, index=False)
print(f'Dataset exportado a {output_path}')
print(f'Filas: {len(df_final)}, Columnas: {len(df_final.columns)}')


Dataset exportado a ../data/df_final_powerbi.csv
Filas: 850, Columnas: 48
